# 01 · Pierwsze dwa pliki
**Cel:** odsłuchać, sprawdzić jakość i zobaczyć różnice. Około 10 minut.

Domyślnie otwierasz **symulację**, nie pomiary klienta. Zmień tylko komórkę KONFIGURACJA. Pracujemy lokalnie. `Shift+Enter` uruchamia komórkę; **Run → Run All Cells** uruchamia całość.

In [1]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Uruchom notebook z katalogu repo lub notebooks/."
os.chdir(ROOT)
from pinglab.audio import Config, load_audio, detect_hits, cut_hits, listen
from pinglab.demo import generate_demo
if not (ROOT / "data/demo/SYN_001_ok.wav").exists():
    generate_demo(ROOT / "data/demo")
CONFIG = Config()
FMAX = 20000  # wspólne pasmo wszystkich plików; dla źródeł 16 kHz ustaw np. 7500
print("Repo:", ROOT)


Repo: /Users/karolsii/dev/ceramic-ping-lab


## Wrzuć pliki
Opcjonalny przycisk zapisze je do `data/raw/inbox/`. Możesz też skopiować je Finderem. Wklej ścieżki poniżej.

In [2]:
from pinglab.widgets import upload_box
uploader = upload_box(ROOT / "data/raw/inbox")

In [3]:
# KONFIGURACJA — jedyne obowiązkowe zmiany po otrzymaniu plików
FILE_A = ROOT / "data/demo/SYN_001_ok.wav"
FILE_B = ROOT / "data/demo/SYN_021_nok.wav"
NAME_A, NAME_B = "A · demo OK", "B · demo NOK"
CHANNEL_A, CHANNEL_B = None, None  # None = kanał z najwyższym RMS; 0/1 wymusza kanał
MANUAL_A, MANUAL_B = None, None  # np. [0.4, 1.9, 3.4]; [] = brak uderzeń
HIT_A, HIT_B = 0, 0             # numer uderzenia od zera


In [4]:
a = load_audio(FILE_A, CONFIG, channel=CHANNEL_A)
b = load_audio(FILE_B, CONFIG, channel=CHANNEL_B)
display(pd.DataFrame([a.qc,b.qc], index=[NAME_A,NAME_B]).T)
for name, audio in [(NAME_A,a),(NAME_B,b)]:
    print(name, "· głośność oryginalna")
    display(listen(audio.y,audio.sr,normalize=False))

,A · demo OK,B · demo NOK
duration_s,5.0,5.0
native_sr,48000,48000
channels,1,1
selected_channel,0,0
peak_dbfs,-3.956122,-3.473422
rms_dbfs,-24.082506,-27.890316
clipping_fraction,0.0,0.0
dc_offset,-0.000006,-0.000015
warnings,[],[]


A · demo OK · głośność oryginalna


B · demo NOK · głośność oryginalna


**Co sprawdzić:** przesterowanie, mowa podczas stuknięcia, dźwięk dotknięcia mikrofonu, za krótki ogon. `dBFS` mierzy poziom cyfrowy, nie głośność w fabryce. Mała amplituda nie oznacza automatycznie złej płytki.

In [5]:
from pinglab.plots import inspection, compare_hits
onsets_a = detect_hits(a.y,a.sr,CONFIG) if MANUAL_A is None else np.array(MANUAL_A)
onsets_b = detect_hits(b.y,b.sr,CONFIG) if MANUAL_B is None else np.array(MANUAL_B)
print("A:", onsets_a, "B:", onsets_b)
display(inspection(a,onsets_a))
display(inspection(b,onsets_b))

A: [0.395 1.895 3.395] B: [0.395 1.895 3.395]


## Porównaj podobne miejsca uderzenia
Jeśli automat podzielił źle: wpisz poprawne sekundy do `MANUAL_A/B` i uruchom ponownie. Nie traktuj każdego głośnego zdarzenia jak uderzenia w ceramikę. Poniższy odsłuch wyrównuje szczyty: służy do porównania barwy, nie głośności.

In [6]:
clips_a = cut_hits(a.y,a.sr,onsets_a,CONFIG)
clips_b = cut_hits(b.y,b.sr,onsets_b,CONFIG)
assert clips_a and clips_b, "Brak uderzeń. Uzupełnij MANUAL_A / MANUAL_B."
ca,cb = clips_a[HIT_A],clips_b[HIT_B]
for name,c in [(NAME_A,ca),(NAME_B,cb)]:
    print(name, "· wyrównany szczyt", "· skrócone okno" if c['truncated'] else "")
    display(listen(c['y'],CONFIG.sr,normalize=True))
display(compare_hits([ca['y'],cb['y']],CONFIG.sr,[NAME_A,NAME_B],
                     [ca['onset_offset_s'],cb['onset_offset_s']],FMAX))

A · demo OK · wyrównany szczyt 


B · demo NOK · wyrównany szczyt 


In [7]:
from pinglab.features import extract_features
assert FMAX <= min(a.native_sr,b.native_sr)/2, "Obniż FMAX do wspólnego pasma źródeł."
fa = extract_features(ca['y'],CONFIG.sr,ca['onset_offset_s'],FMAX)
fb = extract_features(cb['y'],CONFIG.sr,cb['onset_offset_s'],FMAX)
keys=['peak_hz','centroid_hz','bandwidth_hz','decay10_s','late_early_ratio','flatness']
display(pd.DataFrame({NAME_A:fa,NAME_B:fb}).loc[keys].round(3))

,A · demo OK,B · demo NOK
peak_hz,1804.688,1570.312
centroid_hz,2219.773,1890.857
bandwidth_hz,872.507,735.460
decay10_s,0.240,0.080
late_early_ratio,0.001,0.000
flatness,0.000,0.001


## Feedback dla zespołu pomiarowego
Uzupełnij trzy zdania: **Jakość:** … **Różnica A/B:** … **Następne nagranie:** …

Przesunięty rezonans lub krótsze wybrzmiewanie to trop. Mogą też wynikać z podparcia, geometrii, miejsca uderzenia albo mikrofonu. Z dwóch plików nie licz skuteczności. Dalej: **02** — cała seria i korekta podziału.